# LLM Profile Diagnostic

Produces a structured diagnostic for one customer, for an energy **consultant** preparing to
advise that customer - not for the end customer directly (that's a later phase of this project).

This runs on the **single test customer** generated in `00_synthetic_customer_generator.ipynb` and processed in `02_peaks_and_features.ipynb` - not the bulk training population, which exists only to train K-Means (see `03_kmeans_clustering.ipynb`). Using notebook 02's precomputed features/peaks (via `diagnose_customer_from_features()`, rather than `diagnose_customer()`'s raw-data path) keeps this notebook and `04_financial_analysis.ipynb` looking at the same customer, and skips recomputing peaks/features notebook 02 already produced. The app itself still uses `diagnose_customer()` on raw uploaded data - see `diagnosis.py` - this notebook just doesn't need to repeat that computation for a customer notebook 02 already processed.

## Governing principle

**Python owns the truth. The LLM owns the explanation.** Every number in the diagnostic comes from
this project's own pipeline (behavioral features, peak detection, cluster assignment, financial
analysis). The LLM never calculates anything - it interprets and explains what's already been
computed, and is required to say so explicitly when something isn't available, rather than
inventing or working around it.

## No live network access in this sandbox

This environment cannot make outbound API calls. Everything up to the actual API call is built and
tested here - input construction, schema validation, prompt content - plus one **hand-written
illustrative example** (clearly labeled, not LLM output, and not tied to whichever customer this
run happens to generate - see that section for why) so you can react to tone and clarity before
spending any real API calls. The live call cell is guarded behind a flag and is meant to be run in
your own environment once you have an API key configured.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "../src")
import synthetic_generator as sg
import diagnosis as dx
import llm_profile as lp
import json

OUTPUTS_DIR = Path("../data/outputs")
PROFILES_DIR = OUTPUTS_DIR / "llm_profiles"
PROFILES_DIR.mkdir(parents=True, exist_ok=True)
import pandas as pd


## Load the single test customer and build the record

Loads the customer generated by `00_synthetic_customer_generator.ipynb` and its precomputed peaks/features from `02_peaks_and_features.ipynb` (via the `LATEST_CUSTOMER_ID.txt` pointer file), then calls `diagnose_customer_from_features()` - the same logic `diagnose_customer()` uses from here onward (cluster prediction → financial cost → peak-reduction scenarios → record assembly), just starting from already-computed features/peaks instead of raw electricity data. Re-run 00 (then 02) first for a different customer.

In [ ]:
SINGLE_TEST_DIR = Path("../data/inputs/generated/single_test")
CUSTOMER_ID = (SINGLE_TEST_DIR / "LATEST_CUSTOMER_ID.txt").read_text().strip()

customer_features = pd.read_csv(SINGLE_TEST_DIR / f"{CUSTOMER_ID}_electricity_features.csv")
customer_peaks = pd.read_csv(SINGLE_TEST_DIR / f"{CUSTOMER_ID}_electricity_peaks.csv")

record = dx.diagnose_customer_from_features(CUSTOMER_ID, customer_features, customer_peaks)
print("customer_id:", CUSTOMER_ID)
print(json.dumps(record, indent=2, default=str))


## The diagnostic schema and system prompt

Review these before spending any API calls - they define everything the LLM is allowed and required to do.

In [ ]:
print(json.dumps(lp.PROFILE_SCHEMA, indent=2))


In [ ]:
print(lp.SYSTEM_PROMPT)


## Illustrative example (hand-written, NOT generated by an LLM)

A **fixed reference example** - not the customer generated above, and not regenerated each run.
Its purpose is to demonstrate tone and clarity in the abstract, grounded in one real, previously
computed customer's actual numbers (so it's honest, not fabricated) - not to preview what this
specific run's live call will produce. If this isn't the right style, tell me what to change and
I'll adjust `SYSTEM_PROMPT` before we go live.

In [ ]:
illustrative_example = {
    "customer_id": "NRW_COMP_0010",
    "profile_summary": (
        "This customer consumes 1,488,659 kWh annually with a peak demand of 613.7 kW and a "
        "load factor of 0.28. It currently sits in the low-utilization tariff bracket "
        "(2,425.9 h/a - only about 3% below the 2,500 h/a threshold), where 90% of its "
        "€108,789 annual network cost comes from the energy charge rather than the demand charge."
    ),
    "behavior_summary": (
        "Consumption is exclusively a weekday pattern (77.7% of annual energy on weekdays, "
        "22.3% on weekends) but is spread fairly evenly across the day rather than concentrated "
        "in one window: 38.4% during daytime hours, 21.6% in the evening, 19.1% in the morning, "
        "and 15.7% overnight. The coefficient of variation (0.43) and load factor (0.28) both "
        "point to a customer with real but moderate variability - not as extreme as some peaky "
        "customers, but well below a flat, steady-state load."
    ),
    "main_cost_driver": (
        "Energy charge (Arbeitspreis): €97,805 of the €108,789 total annual cost (about 90%) "
        "comes from per-kWh energy consumption rather than the €10,985 demand charge - a direct "
        "consequence of sitting in the low-utilization tariff bracket, which trades a lower "
        "demand rate for a substantially higher energy rate."
    ),
    "key_findings": [
        "Utilization hours (2,425.9 h/a) sit just below the 2,500 h/a tariff threshold - only about 74 hours short - putting this customer on the low-utilization bracket's much higher energy rate (6.57 ct/kWh vs. 1.25 ct/kWh above the threshold).",
        "Only 12 peak events were detected over the year - fewer than typical in this dataset - with a mean peak ratio of 2.87x and maximum of 3.41x baseline.",
        "Peak events are split across daytime (50%), morning (25%), and evening (25%) - unlike customers whose peaks concentrate in a single time-of-day window, this customer's peaks aren't tied to one obvious operating window.",
        "All 12 peak events occurred on weekdays, none on weekends - but one of the three highest (541.1 kW on 2026-12-25) falls on a public holiday that happens to be a weekday, worth confirming whether that reflects genuine business activity or an unattended/automated process.",
        "Every tested peak-reduction scenario (5% through 30%) pushes this customer across the 2,500 h/a threshold into the high-utilization bracket - because the customer already sits so close to it, there is no 'safe' small reduction that avoids the bracket change.",
        "Savings scale favorably with the size of the reduction: a 5% peak cut saves only 2.03% of annual cost, but a 30% cut saves 23.31% - the proportional benefit grows because the higher post-crossing demand-charge rate is a fixed cost of crossing at all, so a bigger peak cut earns more energy-charge savings against that same fixed cost.",
    ],
    "investigation_pointers": [
        "Peak events spread fairly evenly across morning, daytime, and evening - rather than concentrated in one window - often indicate multiple independent processes or pieces of equipment operating on separate schedules, rather than a single identifiable driver. Worth mapping which systems are actually active during each peak window.",
        "A peak recorded on a date when normal business operations are typically closed is a common sign of an unattended automated process (e.g. HVAC or a scheduled batch job not accounting for the calendar) - often inexpensive to address once identified, since the fix is scheduling rather than new equipment.",
    ],
    "consultant_focus": [
        "Discuss the bracket-crossing dynamic directly: this customer is close enough to the 2,500 h/a threshold that any peak-reduction effort will change their tariff structure, not just their peak - the size of the effort should be chosen with that in mind, not treated as a side effect.",
        "Ask what drives the roughly even morning/daytime/evening split in peak timing - unlike a single fixed operating window, this suggests either multiple distinct processes or genuinely variable scheduling.",
        "Follow up on the 2026-12-25 peak specifically.",
    ],
    "caveats": [
        "Only 12 peak events were detected for this customer - smaller than the typical count in this dataset - so the time-of-day peak-share breakdown (25% morning / 50% daytime / 25% evening) is based on a small sample and could shift with more data.",
        "All five tested reduction scenarios happen to cross the tariff bracket for this customer; this is a property of how close their baseline utilization hours are to the threshold, not a general finding about peak reduction.",
    ],
}

problems = lp.validate_profile(illustrative_example)
print("Schema validation problems (should be empty):", problems)
print()
print(json.dumps(illustrative_example, indent=2))


## Live API call (requires your own API key - guarded, won't run here)

Generates the diagnosis for the customer generated above (`record`), not the illustrative
example. This sandbox has no network access, so this cell is guarded behind `RUN_LIVE_CALL`. Set
it to `True` and set `OPENAI_API_KEY` in your environment once you're ready to test with a real
key.

In [ ]:
# Loads OPENAI_API_KEY from a local .env file if present, so the key never
# needs to be pasted into this notebook or committed anywhere - create a file
# named ".env" next to this notebook containing one line:
#   OPENAI_API_KEY=sk-...
try:
    from dotenv import load_dotenv
    load_dotenv(override=True)  # override=True: a fresh .env value always wins over
                                 # anything already in the environment (a stale shell
                                 # var, an old placeholder) - the default silently
                                 # keeps the OLD value otherwise
except Exception:
    pass  # fine either way - python-dotenv missing, no .env file, or any other
          # environment quirk all fall back cleanly to OPENAI_API_KEY already set
          # in your shell/Colab environment

RUN_LIVE_CALL = False  # flip to True once OPENAI_API_KEY is available (via .env or your environment)

if RUN_LIVE_CALL:
    diagnosis = lp.generate_profile(record)
    print(json.dumps(diagnosis, indent=2))

    out_path = PROFILES_DIR / f"{CUSTOMER_ID}.json"
    out_path.write_text(json.dumps(diagnosis, indent=2))
    print("\nSaved:", out_path)
else:
    print("RUN_LIVE_CALL is False - skipping. Set it to True with an API key configured to test.")
